# 鳩の目、京都。 実 GPU での動作計測（Google Colab）

作品を NVIDIA GPU 付きのランタイムで開き、視点・時刻・人の種類ごとに、作品自身が測っている値（1 コマの時間、内部解像度、画質、描画命令数、三角形数）とスクリーンショットを記録します。

**使い方**

1. メニューの「ランタイム」→「ランタイムのタイプを変更」で、ハードウェア アクセラレータを **T4 GPU** にして保存する。
2. 「ランタイム」→「すべてのセルを実行」。10〜20 分ほどかかります。
3. 最後のセルに出る表（Markdown）を、そのまま GitHub の Issue に貼る。

**分かること・分からないこと**

- 分かる: 実際の GPU とそのドライバーでの速度、視点ごとの重さの差、`?people=asset` の重さ、描画エラーの有無。
- 分からない: スマートフォンでの動作（これはデータセンター用 GPU です）、音の聴感。

GPU を掴む手順は Chrome 公式の解説（<https://developer.chrome.com/docs/web-platform/webgpu/colab-headless>）に沿っています。Colab 既定のドライバーのままでは Chrome が GPU を認識しないため、Vulkan 用のライブラリを入れ直します。

## 1. GPU が割り当てられているか

In [ ]:
import subprocess, shutil
if not shutil.which('nvidia-smi'):
    raise SystemExit('GPU がありません。「ランタイム」→「ランタイムのタイプを変更」で T4 GPU を選んでから、もう一度すべて実行してください。')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
DRIVER = subprocess.run(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip().splitlines()[0]
MAJOR = DRIVER.split('.')[0]
print('driver', DRIVER, '-> libnvidia-gl-' + MAJOR)

## 2. Vulkan のライブラリと Chrome を入れる

ドライバーと同じ番号の `libnvidia-gl` を入れます。番号が合うパッケージが無い場合はここで失敗するので、出力をそのまま Issue に貼ってください。

In [ ]:
!apt-get -qq update
!apt-get -qq install -y vulkan-tools libnvidia-gl-{MAJOR} > /dev/null
!vulkaninfo --summary 2>/dev/null | grep -i -E "deviceName|deviceType|driverName|driverInfo" || echo "vulkaninfo が GPU を表示しません"
!wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get -qq install -y /tmp/chrome.deb > /dev/null
!google-chrome-stable --version
!pip -q install playwright

## 3. 作品を取得する

In [ ]:
import os
if not os.path.isdir('/content/hato-no-me-kyoto'):
    !git clone -q --depth 1 https://github.com/kai-kou/hato-no-me-kyoto.git /content/hato-no-me-kyoto
!cd /content/hato-no-me-kyoto && git log --oneline -1 && du -sh app

## 4. 計測する

Chrome の起動方法を Vulkan → EGL → 既定 の順に試し、ソフトウェア描画にならなかった最初のもので全項目を測ります。どれでも GPU を使えなかった場合は、その旨が結果に書かれます（そのときの数値は実 GPU の値ではありません）。

In [ ]:
import json, subprocess, os
env = dict(os.environ, HATO_CHROME='/usr/bin/google-chrome-stable')
OUT = '/content/gpucheck'
chosen = None
for backend in ['vulkan', 'egl', 'default']:
    # a quick probe first: one light case tells whether this way of starting Chrome reaches the GPU
    r = subprocess.run(['python3', 'tools/gpucheck.py', '--backend', backend, '--out', OUT + '_probe', '--modes', 'tele', '--hours', '11', '--people', 'code', '--settle', '2', '--samples', '2'],
                       cwd='/content/hato-no-me-kyoto', env=env, capture_output=True, text=True)
    print('[' + backend + ']', (r.stdout.strip().splitlines() or ['(no output)'])[0]); 
    if r.returncode != 0: print(r.stderr[-1500:]); continue
    if not json.load(open(OUT + '_probe/results.json'))['software']: chosen = backend; break
if chosen is None:
    print('どの起動方法でも GPU を使えませんでした。ソフトウェア描画のまま計測します。'); chosen = 'vulkan'
print('計測に使う起動方法:', chosen)
!cd /content/hato-no-me-kyoto && HATO_CHROME=/usr/bin/google-chrome-stable python3 tools/gpucheck.py --backend {chosen} --out {OUT}

## 5. 結果

下の枠の中身（Markdown）を Issue にそのまま貼れます。

In [ ]:
from IPython.display import Markdown, display, Image
txt = open('/content/gpucheck/results.md', encoding='utf-8').read()
display(Markdown(txt))
print('-' * 30, 'ここから下をコピー', '-' * 30)
print(txt)

In [ ]:
# スクリーンショット（形の崩れ、真っ黒、ちらつきが無いかを目で見る用）
import json
from IPython.display import Image, display
R = json.load(open('/content/gpucheck/results.json'))
for c in R['cases']:
    if c.get('shot'):
        print(c['people'], 't=' + c['hour'], c['mode'], c.get('ms_median'), 'ms')
        display(Image('/content/gpucheck/' + c['shot'], width=720))

In [ ]:
# まとめてダウンロードする場合
!cd /content && zip -q -r gpucheck.zip gpucheck
from google.colab import files
files.download('/content/gpucheck.zip')